# Structural Coherence Feature Extraction

This notebook extracts only the structural sentence-transformers sub-extractor output: `data/features/structural/coherence_<split>.parquet`.

It computes the 4 coherence dimensions from `sentence-transformers/all-MiniLM-L6-v2`: mean consecutive coherence, std consecutive coherence, topic drift, and coherence break rate.

In [ ]:
!nvidia-smi
!pip -q install sentence-transformers pandas pyarrow tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Change this path to your copied project folder in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/mental_health_fusion')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUT_DIR = PROJECT_DIR / 'data' / 'features' / 'structural'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = 'text'
POST_ID_COL = 'post_id'
BATCH_SIZE = 64
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print('Project:', PROJECT_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SENTENCE_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return [s.strip() for s in SENTENCE_RE.split(text.strip()) if s.strip()]

def cosine(a, b):
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)

def post_ids_for(df, split):
    if POST_ID_COL in df.columns:
        return df[POST_ID_COL].astype(str).tolist()
    return [f'{split}_{i}' for i in range(len(df))]

model = SentenceTransformer(MODEL_NAME)
print('Model device:', model.device)

In [ ]:
def extract_coherence_one(text):
    sentences = split_sentences(text)
    if len(sentences) < 2:
        return np.zeros(4, dtype=np.float32)

    embeddings = np.asarray(
        model.encode(
            sentences,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
        ),
        dtype=np.float32,
    )
    sims = np.asarray(
        [cosine(embeddings[i], embeddings[i + 1]) for i in range(len(embeddings) - 1)],
        dtype=np.float32,
    )
    return np.asarray(
        [
            float(sims.mean()),
            float(sims.std(ddof=0)),
            cosine(embeddings[0], embeddings[-1]),
            float(np.mean(sims < 0.3)),
        ],
        dtype=np.float32,
    )

def extract_split(split, force=False):
    input_path = PROCESSED_DIR / f'{split}.csv'
    output_path = OUTPUT_DIR / f'coherence_{split}.parquet'
    if output_path.exists() and not force:
        print(f'Skipping existing: {output_path}')
        return output_path
    if not input_path.exists():
        raise FileNotFoundError(f'Missing input CSV: {input_path}')

    df = pd.read_csv(input_path)
    if TEXT_COL not in df.columns:
        raise KeyError(f'Missing text column {TEXT_COL!r} in {input_path}')

    texts = df[TEXT_COL].fillna('').astype(str).tolist()
    features = [extract_coherence_one(text) for text in tqdm(texts, desc=f'coherence {split}')]
    out = pd.DataFrame({'post_id': post_ids_for(df, split), 'features': features})
    out.to_parquet(output_path, index=False)
    print(f'Saved {len(out)} rows -> {output_path}')
    return output_path

In [ ]:
# Start with val to confirm the pipeline before running train/test.
extract_split('val', force=False)
extract_split('train', force=False)
extract_split('test', force=False)

In [ ]:
for split in ['train', 'val', 'test']:
    path = OUTPUT_DIR / f'coherence_{split}.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        dims = {len(x) for x in df['features']}
        print(split, len(df), dims)
        assert dims == {4}
        assert np.all(np.isfinite(np.vstack(df['features'].to_numpy())))